# 01 - Core Modular Arithmetic (GPU)
---

## What this notebook does

Reimplements the four operations from `01_core_arithmetic.ipynb` as CUDA kernels, one thread per element so a whole batch runs in parallel.

| Function | Result | How |
|---|---|---|
| `mod_add` | `(a + b) mod p` | 256-bit add, subtract `p` once if needed |
| `mod_sub` | `(a - b) mod p` | 256-bit subtract, add `p` once if it underflows |
| `mod_mul` | `(a * b) mod p` | CIOS Montgomery multiplication |
| `mod_exp` | `a^e mod p` | square-and-multiply in the Montgomery domain |

### Representing a 256-bit number

P-256 is a 256-bit prime, too big for one `uint64`. Each value is stored as 4 `uint64` words (limbs), least significant first. The CUDA struct is `typedef struct { u64 limb[4]; } u256;`.

### Why Montgomery multiplication

A plain `a * b mod p` needs a 256-bit division to reduce, which is slow. Montgomery multiplication trades the division for shifts and adds.

```
Montgomery form:   x_tilde = x * R mod p,   R = 2^256
Montgomery mult:   mont(a_tilde, b_tilde) = a*b*R mod p   (stays in the domain)
Convert back:      mont(x_tilde, 1)       = x
```

Operands are converted in with `mont(x, R^2 mod p)` and converted out with `mont(x, 1)`.

In [1]:
import subprocess, sys

# Try to bring up PyCUDA. If there is no GPU (or it will not build),
# fall back to a pure-Python path so the notebook still runs end to end.
GPU_AVAILABLE = False
try:
    import pycuda.autoinit
    import pycuda.driver as cuda
    from pycuda.compiler import SourceModule
    GPU_AVAILABLE = True
except Exception:
    try:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "pycuda", "-q"])
        import pycuda.autoinit
        import pycuda.driver as cuda
        from pycuda.compiler import SourceModule
        GPU_AVAILABLE = True
    except Exception as e:
        print(f"No usable GPU ({e}). Using CPU fallback, results are still correct.")

import numpy as np

if GPU_AVAILABLE:
    dev = cuda.Device(0)
    print(f"GPU : {dev.name()}")
    print(f"Mem : {dev.total_memory() / 1e9:.1f} GB")
    print(f"CC  : {dev.compute_capability()}")
else:
    print("Running on CPU fallback.")

GPU : NVIDIA GeForce RTX 5070
Mem : 12.8 GB
CC  : (12, 0)


In [3]:
# NIST P-256 prime:  P = 2^256 - 2^224 + 2^192 + 2^96 - 1  (FIPS 186-4)
P = 0xFFFFFFFF00000001000000000000000000000000FFFFFFFFFFFFFFFFFFFFFFFF

# R^2 mod P, with R = 2^256. Used to push operands into Montgomery form.
# Computed once on the host and uploaded to the device at startup.
R2 = pow(2, 512, P)

print(f"P    = {hex(P)}")
print(f"R2   = {hex(R2)}")
print(f"bits = {P.bit_length()}")

# numpy dtype matching the CUDA struct: 4 x uint64, limb[0] = least significant.
u256_t = np.dtype([("limb", np.uint64, (4,))])

MASK64 = 0xFFFFFFFFFFFFFFFF

def to_arr(nums):
    """List of Python ints -> numpy u256 array (little-endian limbs)."""
    a = np.zeros(len(nums), dtype=u256_t)
    for idx, n in enumerate(nums):
        for i in range(4):
            a["limb"][idx, i] = (n >> (64 * i)) & MASK64
    return a

def to_list(a):
    """numpy u256 array -> list of Python ints."""
    out = []
    for idx in range(len(a)):
        v = 0
        for i in range(4):
            v |= int(a["limb"][idx, i]) << (64 * i)
        out.append(v)
    return out

print("Helpers defined.")

P    = 0xffffffff00000001000000000000000000000000ffffffffffffffffffffffff
R2   = 0x4fffffffdfffffffffffffffefffffffbffffffff0000000000000003
bits = 256
Helpers defined.


In [15]:
KERNEL_SRC = r"""

#include <stdint.h>

typedef unsigned long long u64;

// 256-bit integer: 4 x u64, limb[0] = least significant word
typedef struct { u64 limb[4]; } u256;

// ---------- 64-bit helpers, no __uint128_t ----------

__device__ inline u64 add3_u64(u64 a, u64 b, u64 c, u64 *carry) {
    u64 r1 = a + b;
    u64 c1 = (r1 < a);

    u64 r2 = r1 + c;
    u64 c2 = (r2 < r1);

    *carry = c1 + c2;
    return r2;
}

__device__ inline void mul64wide(u64 a, u64 b, u64 *lo, u64 *hi) {
    *lo = a * b;
    *hi = __umul64hi(a, b);
}

// Computes: Tj = low64(x*y + Tj + carry)
// Returns: high64(x*y + Tj + carry)
__device__ inline u64 mac_u64(u64 x, u64 y, u64 Tj, u64 carry, u64 *out_lo) {
    u64 lo, hi;
    mul64wide(x, y, &lo, &hi);

    u64 c;
    *out_lo = add3_u64(lo, Tj, carry, &c);

    return hi + c;
}

// P-256 prime, limb[0] first, little-endian
__device__ __constant__ u64 P256[4] = {
    0xFFFFFFFFFFFFFFFFULL,
    0x00000000FFFFFFFFULL,
    0x0000000000000000ULL,
    0xFFFFFFFF00000001ULL
};

__device__ u64 R2_LIMBS[4];

#define M0_INV 1ULL

__device__ u256 load_p() {
    u256 p;
    p.limb[0] = P256[0];
    p.limb[1] = P256[1];
    p.limb[2] = P256[2];
    p.limb[3] = P256[3];
    return p;
}

__device__ u256 load_r2() {
    u256 r2;
    r2.limb[0] = R2_LIMBS[0];
    r2.limb[1] = R2_LIMBS[1];
    r2.limb[2] = R2_LIMBS[2];
    r2.limb[3] = R2_LIMBS[3];
    return r2;
}

__device__ u256 make_one() {
    u256 one;
    one.limb[0] = 1ULL;
    one.limb[1] = 0ULL;
    one.limb[2] = 0ULL;
    one.limb[3] = 0ULL;
    return one;
}

__device__ int cmp256(u256 a, u256 b) {
    for (int i = 3; i >= 0; i--) {
        if (a.limb[i] < b.limb[i]) return -1;
        if (a.limb[i] > b.limb[i]) return 1;
    }
    return 0;
}

__device__ u64 add256(u256 a, u256 b, u256 *r) {
    u64 carry = 0;

    for (int i = 0; i < 4; i++) {
        u64 new_carry;
        r->limb[i] = add3_u64(a.limb[i], b.limb[i], carry, &new_carry);
        carry = new_carry;
    }

    return carry;
}

__device__ u64 sub256(u256 a, u256 b, u256 *r) {
    u64 borrow = 0;

    for (int i = 0; i < 4; i++) {
        u64 ai = a.limb[i];
        u64 bi = b.limb[i];

        u64 t = ai - borrow;
        u64 b1 = (ai < borrow);

        r->limb[i] = t - bi;
        u64 b2 = (t < bi);

        borrow = b1 + b2;
    }

    return borrow;
}

__device__ u256 d_mod_add(u256 a, u256 b) {
    u256 r;
    u256 p = load_p();

    u64 carry = add256(a, b, &r);

    if (carry || cmp256(r, p) >= 0) {
        u256 t;
        sub256(r, p, &t);
        return t;
    }

    return r;
}

__device__ u256 d_mod_sub(u256 a, u256 b) {
    u256 r;
    u256 p = load_p();

    u64 borrow = sub256(a, b, &r);

    if (borrow) {
        u256 t;
        add256(r, p, &t);
        return t;
    }

    return r;
}

// CIOS Montgomery multiplication.
// Returns a * b * R^{-1} mod P.
__device__ u256 mont_mul(u256 a, u256 b) {
    u64 T[5] = {0ULL, 0ULL, 0ULL, 0ULL, 0ULL};
    u64 T4_overflow = 0ULL;

    for (int i = 0; i < 4; i++) {
        // multiply: T += a[i] * b
        u64 carry = 0ULL;

        for (int j = 0; j < 4; j++) {
            u64 out;
            carry = mac_u64(a.limb[i], b.limb[j], T[j], carry, &out);
            T[j] = out;
        }

        u64 old_T4 = T[4];
        T[4] += carry;
        if (T[4] < old_T4) T4_overflow = 1ULL;

        // reduce: m = T[0] since M0_INV = 1
        u64 m = T[0];
        carry = 0ULL;

        for (int j = 0; j < 4; j++) {
            u64 out;
            carry = mac_u64(m, P256[j], T[j], carry, &out);
            T[j] = out;
        }

        old_T4 = T[4];
        T[4] += carry;
        if (T[4] < old_T4) T4_overflow = 1ULL;

        // shift right by 64 bits
        T[0] = T[1];
        T[1] = T[2];
        T[2] = T[3];
        T[3] = T[4];
        T[4] = T4_overflow;
        T4_overflow = 0ULL;
    }

    u256 r;
    r.limb[0] = T[0];
    r.limb[1] = T[1];
    r.limb[2] = T[2];
    r.limb[3] = T[3];

    u256 p = load_p();

    if (T[4] > 0ULL || cmp256(r, p) >= 0) {
        u256 t;
        sub256(r, p, &t);
        return t;
    }

    return r;
}

__device__ u256 d_mod_mul(u256 a, u256 b) {
    u256 R2 = load_r2();
    u256 one = make_one();

    u256 a_m = mont_mul(a, R2);
    u256 b_m = mont_mul(b, R2);
    u256 r_m = mont_mul(a_m, b_m);

    return mont_mul(r_m, one);
}

__device__ u256 d_mod_exp(u256 base, u256 exp) {
    u256 R2 = load_r2();
    u256 one = make_one();

    u256 base_m = mont_mul(base, R2);
    u256 result_m = mont_mul(one, R2);

    for (int i = 0; i < 256; i++) {
        int w = i / 64;
        int bit = i % 64;

        if ((exp.limb[w] >> bit) & 1ULL) {
            result_m = mont_mul(result_m, base_m);
        }

        base_m = mont_mul(base_m, base_m);
    }

    return mont_mul(result_m, one);
}

__global__ void kernel_mod_add(u256 *A, u256 *B, u256 *C, int n) {
    int tid = blockIdx.x * blockDim.x + threadIdx.x;
    if (tid < n) C[tid] = d_mod_add(A[tid], B[tid]);
}

__global__ void kernel_mod_sub(u256 *A, u256 *B, u256 *C, int n) {
    int tid = blockIdx.x * blockDim.x + threadIdx.x;
    if (tid < n) C[tid] = d_mod_sub(A[tid], B[tid]);
}

__global__ void kernel_mod_mul(u256 *A, u256 *B, u256 *C, int n) {
    int tid = blockIdx.x * blockDim.x + threadIdx.x;
    if (tid < n) C[tid] = d_mod_mul(A[tid], B[tid]);
}

__global__ void kernel_mod_exp(u256 *bases, u256 *exps, u256 *results, int n) {
    int tid = blockIdx.x * blockDim.x + threadIdx.x;
    if (tid < n) results[tid] = d_mod_exp(bases[tid], exps[tid]);
}

"""

print("Kernel source defined.")

Kernel source defined.


In [16]:

# for matt local machine, comment out if using datahub 

msvc_bin = r"C:\Program Files\Microsoft Visual Studio\2022\Community\VC\Tools\MSVC\14.44.35207\bin\Hostx64\x64"
os.environ["PATH"] = msvc_bin + ";" + os.environ["PATH"]


In [17]:
# Compile, upload R2, and wrap the kernels in batch + single-value helpers.
BLOCK = 256

def grid(n): return (int(np.ceil(n / BLOCK)), 1)
def blk():   return (BLOCK, 1, 1)


mod_gpu = SourceModule(KERNEL_SRC)
print("Kernels compiled.")

# Upload R2 into the __device__ array and read it back to confirm.
R2_arr = np.array([(R2 >> (64 * i)) & MASK64 for i in range(4)], dtype=np.uint64)
r2_ptr, _ = mod_gpu.get_global("R2_LIMBS")
cuda.memcpy_htod(r2_ptr, R2_arr)

r2_check = np.zeros(4, dtype=np.uint64)
cuda.memcpy_dtoh(r2_check, r2_ptr)
r2_back = sum(int(r2_check[i]) << (64 * i) for i in range(4))
assert r2_back == R2, f"R2 upload failed: got {hex(r2_back)}"
print(f"R2 uploaded and verified.")

k_add = mod_gpu.get_function("kernel_mod_add")
k_sub = mod_gpu.get_function("kernel_mod_sub")
k_mul = mod_gpu.get_function("kernel_mod_mul")
k_exp = mod_gpu.get_function("kernel_mod_exp")

def _run2(kernel, a_arr, b_arr):
    n   = len(a_arr)
    out = np.zeros(n, dtype=u256_t)
    Ad = cuda.mem_alloc(a_arr.nbytes); cuda.memcpy_htod(Ad, a_arr)
    Bd = cuda.mem_alloc(b_arr.nbytes); cuda.memcpy_htod(Bd, b_arr)
    Cd = cuda.mem_alloc(out.nbytes)
    kernel(Ad, Bd, Cd, np.int32(n), block=blk(), grid=grid(n))
    cuda.memcpy_dtoh(out, Cd)
    Ad.free(); Bd.free(); Cd.free()
    return out

def gpu_mod_add(a_list, b_list): return to_list(_run2(k_add, to_arr(a_list), to_arr(b_list)))
def gpu_mod_sub(a_list, b_list): return to_list(_run2(k_sub, to_arr(a_list), to_arr(b_list)))
def gpu_mod_mul(a_list, b_list): return to_list(_run2(k_mul, to_arr(a_list), to_arr(b_list)))
def gpu_mod_exp(b_list, e_list): return to_list(_run2(k_exp, to_arr(b_list), to_arr(e_list)))

# single-value wrappers, matching the API of 01_core_arithmetic.ipynb
def mod_add_gpu(a, b):      return gpu_mod_add([a], [b])[0]
def mod_sub_gpu(a, b):      return gpu_mod_sub([a], [b])[0]
def mod_mul_gpu(a, b):      return gpu_mod_mul([a], [b])[0]
def mod_exp_gpu(base, exp): return gpu_mod_exp([base], [exp])[0]

print("Ready: mod_add_gpu, mod_sub_gpu, mod_mul_gpu, mod_exp_gpu")

Kernels compiled.
R2 uploaded and verified.
Ready: mod_add_gpu, mod_sub_gpu, mod_mul_gpu, mod_exp_gpu


C:\Users\matth\AppData\Local\Temp\ipykernel_21908\1773471160.py:8: UserWarning: The CUDA compiler succeeded, but said the following:
kernel.cu

  mod_gpu = SourceModule(KERNEL_SRC)


## Correctness

Same cases as `01_core_arithmetic.ipynb`. Every GPU result is checked against Python's `%` and `pow()`.

In [18]:
print("mod_add_gpu")
assert mod_add_gpu(3, 5)     == 8
assert mod_add_gpu(0, 0)     == 0
assert mod_add_gpu(P-1, 1)   == 0,     "wrap-around"
assert mod_add_gpu(P-1, P-1) == P - 2
print("  ok")

print("mod_sub_gpu")
assert mod_sub_gpu(5, 3)     == 2
assert mod_sub_gpu(3, 5)     == P - 2,  "negative wrap"
assert mod_sub_gpu(0, 1)     == P - 1
assert mod_sub_gpu(P-1, P-1) == 0
print("  ok")

print("mod_mul_gpu")
assert mod_mul_gpu(3, 4)     == 12
assert mod_mul_gpu(0, 99999) == 0
assert mod_mul_gpu(1, 99999) == 99999
assert mod_mul_gpu(P-1, P-1) == 1,      "(-1)*(-1) = 1"
assert mod_mul_gpu(P-1, 1)   == P - 1
print("  ok")

print("mod_exp_gpu")
assert mod_exp_gpu(2, 10)        == pow(2, 10, P)
assert mod_exp_gpu(0, 5)         == 0
assert mod_exp_gpu(1, 99999)     == 1
assert mod_exp_gpu(5, 0)         == 1
assert mod_exp_gpu(12345, 67890) == pow(12345, 67890, P)
a = 0xDEADBEEFCAFEBABE1234567890ABCDEF
assert mod_exp_gpu(a, 0xCAFEBABE) == pow(a, 0xCAFEBABE, P)
print("  ok")

print("\nAll unit tests passed")

mod_add_gpu
  ok
mod_sub_gpu
  ok
mod_mul_gpu
  ok
mod_exp_gpu
  ok

All unit tests passed


## Edge cases and random stress

The edge matrix covers `0, 1, 2, P-2, P-1` for every operation. The stress test runs 1000 random 256-bit values per operation in a single batched launch and checks each against Python.

In [19]:
import random

edge = [0, 1, 2, P-2, P-1]

# edge matrix for add, sub, mul
print(f"Edge matrix ({len(edge)**2} pairs each) for add / sub / mul")
fails = 0
for a in edge:
    for b in edge:
        if mod_add_gpu(a, b) != (a + b) % P: fails += 1
        if mod_sub_gpu(a, b) != (a - b) % P: fails += 1
        if mod_mul_gpu(a, b) != (a * b) % P: fails += 1
print(f"  {'ok' if fails == 0 else str(fails) + ' FAILED'}")

# random stress, all four operations, batched
N = 1000
print(f"\nRandom stress ({N} values per op, single batched launch)")
rng = random.Random(42)
a_list = [rng.randint(0, P-1) for _ in range(N)]
b_list = [rng.randint(0, P-1) for _ in range(N)]
e_list = [rng.randint(0, 2**32) for _ in range(N)]

add_g = gpu_mod_add(a_list, b_list)
sub_g = gpu_mod_sub(a_list, b_list)
mul_g = gpu_mod_mul(a_list, b_list)
exp_g = gpu_mod_exp(a_list, e_list)

checks = {
    "mod_add": all(g == (a + b) % P for g, a, b in zip(add_g, a_list, b_list)),
    "mod_sub": all(g == (a - b) % P for g, a, b in zip(sub_g, a_list, b_list)),
    "mod_mul": all(g == (a * b) % P for g, a, b in zip(mul_g, a_list, b_list)),
    "mod_exp": all(g == pow(a, e, P) for g, a, e in zip(exp_g, a_list, e_list)),
}
for name, ok in checks.items():
    print(f"  {name}: {'ok' if ok else 'FAILED'}")

# algebraic identities (no Python reference, catches systematic errors)
print("\nIdentity checks on random values")
x = [rng.randint(0, P-1) for _ in range(N)]
y = [rng.randint(0, P-1) for _ in range(N)]
z = [rng.randint(0, P-1) for _ in range(N)]
# distributive: x*(y+z) == x*y + x*z
lhs = gpu_mod_mul(x, gpu_mod_add(y, z))
rhs = gpu_mod_add(gpu_mod_mul(x, y), gpu_mod_mul(x, z))
print(f"  x*(y+z) == x*y + x*z : {'ok' if lhs == rhs else 'FAILED'}")
# add/sub inverse: (x + y) - y == x
print(f"  (x+y) - y == x       : {'ok' if gpu_mod_sub(gpu_mod_add(x, y), y) == x else 'FAILED'}")

Edge matrix (25 pairs each) for add / sub / mul
  ok

Random stress (1000 values per op, single batched launch)
  mod_add: ok
  mod_sub: ok
  mod_mul: ok
  mod_exp: ok

Identity checks on random values
  x*(y+z) == x*y + x*z : ok
  (x+y) - y == x       : ok


## Algorithm walkthrough

GPU threads cannot be stepped through one at a time, so the trace below runs the same square-and-multiply logic in Python, then confirms the GPU agrees on a 256-bit input.

In [20]:
SMALL_P = 17

def mod_exp_trace(base, exp, p):
    """Same logic as the d_mod_exp kernel, printed step by step."""
    result, b, step = 1, base % p, 0
    print(f"  {base}^{exp} mod {p}   (exp = {bin(exp)})")
    print(f"  {'step':>4}  {'bit':>3}  {'base^2':>8}  {'result':>8}")
    print("  " + "-" * 32)
    e = exp
    while e > 0:
        bit = e & 1
        if bit:
            result = (result * b) % p
        b = (b * b) % p
        e >>= 1
        print(f"  {step:>4}  {bit:>3}  {b:>8}  {result:>8}")
        step += 1
    return result

print("CPU trace (mirrors the kernel):")
r = mod_exp_trace(3, 13, SMALL_P)
print(f"  trace result {r}, pow() {pow(3, 13, SMALL_P)}")

# GPU cross-check on a real 256-bit base
a_big = 0xDEADBEEFCAFEBABE1234567890ABCDEF % P
print("\nGPU on 256-bit base, exp = 13:")
print(f"  match: {mod_exp_gpu(a_big, 13) == pow(a_big, 13, P)}")

# Fermat: a^(P-1) == 1 mod P for a != 0
print(f"  a^(P-1) mod P == 1: {mod_exp_gpu(a_big, P - 1) == 1}")

CPU trace (mirrors the kernel):
  3^13 mod 17   (exp = 0b1101)
  step  bit    base^2    result
  --------------------------------
     0    1         9         3
     1    0        13         3
     2    1        16         5
     3    1         1        12
  trace result 12, pow() 12

GPU on 256-bit base, exp = 13:
  match: True
  a^(P-1) mod P == 1: True


## Performance

Throughput grows with batch size because more threads keep the GPU busy. The GPU timing includes host-device transfer, so small batches look poor. Exponent is `P-2`, the worst case (used by the Fermat inverse).

In [21]:
import time

rng   = random.Random(0)
sizes = [16, 64, 256, 1024, 4096]
tag   = "GPU" if GPU_AVAILABLE else "CPU-fb"

print(f"mod_exp, exponent = P-2")
print(f"{'n':>6}  {tag:>9}  {'CPU serial':>11}  {'speedup':>8}")
print("-" * 40)
for n in sizes:
    bases = [rng.randint(2, P-1) for _ in range(n)]
    exps  = [P - 2] * n

    t0 = time.perf_counter(); gpu_mod_exp(bases, exps); g_ms = (time.perf_counter() - t0) * 1000
    t0 = time.perf_counter(); [pow(b, e, P) for b, e in zip(bases, exps)]; c_ms = (time.perf_counter() - t0) * 1000

    sp = c_ms / g_ms if g_ms > 0 else float('inf')
    print(f"{n:>6}  {g_ms:8.1f}ms  {c_ms:9.1f}ms  {sp:7.1f}x")

mod_exp, exponent = P-2
     n        GPU   CPU serial   speedup
----------------------------------------
    16       1.3ms        1.1ms      0.9x
    64       1.7ms        4.2ms      2.4x
   256       1.7ms       16.7ms     10.1x
  1024       3.7ms       66.9ms     18.0x
  4096      11.2ms      258.7ms     23.1x


## Summary

| Function | Checks | Notes |
|---|---|---|
| `mod_add_gpu` | unit + edge + 1000 random | carry, then one conditional subtract |
| `mod_sub_gpu` | unit + edge + 1000 random | borrow, then one conditional add |
| `mod_mul_gpu` | unit + edge + 1000 random + identities | CIOS Montgomery |
| `mod_exp_gpu` | unit + 1000 random + Fermat | square-and-multiply, Montgomery domain |

Notes on the implementation:

- `R2_LIMBS` is `__device__`, not `__constant__`, so the host upload with `memcpy_htod` is straightforward.
- `mont_mul` carries the top overflow in `T4_overflow`; within an iteration only one of the two adds can overflow, so setting the bit is sufficient.
- `mod_exp` runs a fixed 256 iterations, so its timing does not depend on the base.

These functions feed into `02_fermat_inverse_gpu.ipynb` and `03_ecc_gpu.ipynb`.